PART 03：动手——142 行，四块代码拼出心脏
整个内核 142 行，拆成四块：工具定义、执行器、主循环、命令行入口。一块一块写。

准备工作（5 分钟）
新建一个目录，两个依赖就够：

In [ ]:
pip install anthropic python-dotenv

然后放一个 .env 文件，三个变量：

In [ ]:
ANTHROPIC_API_KEY=你的Key
MODEL_ID=glm-4.7
# 可选：如果你想用 GLM / DeepSeek / Kimi 的 Anthropic 兼容接口
ANTHROPIC_BASE_URL=https://open.bigmodel.cn/api/anthropic

这里有个对普通人很友好的细节：anthropic 这个 SDK 只是一种消息格式，跟用谁家模型无关。现在主流国产模型都提供 Anthropic 兼容接口，把 ANTHROPIC_BASE_URL 指过去、MODEL_ID 换成对应型号，就能直接跑。不需要非有 Anthropic 账号。

代码里读配置就三行：

In [2]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os
import subprocess

load_dotenv(override=True)
client = Anthropic(base_url=os.getenv("ANTHROPIC_BASE_URL"))
MODEL = os.environ["MODEL_ID"]

In [4]:
# 第一块：工具定义——只给一个 bash
# 先定系统提示，一句话：

SYSTEM = f"You are a coding agent at {os.getcwd()}. Use bash to solve tasks. Act, don't explain."

# 翻译过来：你是一个在我这个目录下干活的编码代理，用 bash 解决问题，行动，别解释。

# 别小看"Act, don't explain"这五个词，它是整个产品性格的起点——少废话，直接干。Claude Code 那种"闷头干活"的气质，源头就在系统提示里。

# 然后是工具定义，全文唯一的工具：

TOOLS = [{
    "name": "bash",
    "description": "Run a shell command.",
    "input_schema": {
        "type": "object",
        "properties": {"command": {"type": "string"}},
        "required": ["command"],
    },
}]


# 一个工具就是三样东西：名字 + 描述 + 参数 schema。模型只看得见 description，它靠这一句"Run a shell command"判断什么时候用、怎么用。

# 就这么点配置，模型就知道自己手里有一把锤子了。

# 第二块：run_bash 执行器——带三条最小安全带
# 模型说要跑命令，总得有人真去跑。执行器长这样：

def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, errors="replace",
                           timeout=120)
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    except (FileNotFoundError, OSError) as e:
        return f"Error: {e}"


# 逻辑很简单：收到命令 → 查黑名单 → 用 subprocess 真实执行 → 把 stdout 和 stderr 拼起来返回。
# 但里面有三条安全带，每条都值得停下来想一想：
# 安全带一：黑名单。 rm -rf /、sudo、shutdown、reboot，四条会直接掀桌子的命令，见到就拦。坦白讲这个名单极其粗糙——rm -rf ~/你的毕业设计 它就拦不住。真正像样的权限系统（审批、白名单、分级放行）是后面的篇章要补的机制，现在先裸奔，所以请一定在临时测试目录里玩。
# 安全带二：120 秒超时。 模型偶尔会生成挂住的命令（比如起了个交互式程序、或者网络请求卡死）。没有超时，你的整个 Agent 就跟着一起死。
# 安全带三：输出截断到 50000 字符。 这条最有味道。记住一个事实：工具的输出会整段进入上下文。模型随手跑一条 find /，输出可能几十万字符，一条命令就能把上下文撑爆，后面的对话全部崩掉。截断是最粗暴但最有效的保险——至于"怎么聪明地压缩"，正是后面"上下文压缩"那篇要解决的问题。
# 这个内核里的每一条防御性代码，背后都对应着后续的一个完整机制。现在是创可贴，以后是器官。
# 第三块：主循环 agent_loop——全文最重要的一段
# 来了，心脏的心脏。先看完整代码，再逐机制拆：



def agent_loop(messages: list):
    while True:
        # 1. 把整本账本发给模型
        response = client.messages.create(
            model=MODEL, system=SYSTEM, messages=messages,
            tools=TOOLS, max_tokens=8000,
        )

        # 2. 模型的回复记进账本
        messages.append({"role": "assistant", "content": response.content})

        # 3. 这轮有没有调工具？没有 → 任务完成，退出
        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        if not tool_calls:
            return

        # 4. 有 → 逐个执行，收集结果
        results = []
        for block in tool_calls:
            print(f"$ {block.input['command']}")
            output = run_bash(block.input["command"])
            print(output[:200])
            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": output,
            })

        # 5. 结果塞回账本，回到第 1 步
        messages.append({"role": "user", "content": results})


# 三十多行，一个 coding agent 的内核就没了。但这段代码里有三个机制，看懂它们，才算看懂 Agent。

# 机制一：messages 是一本"只追加"的账本
# 注意第 2 步和第 5 步：模型的回复 append 进去，工具的结果 append 进去，从头到尾只有 append，没有修改，没有删除。

# messages 这本账本，就是模型的全部记忆。它下一轮做的所有判断——"我刚才干了什么"、"报错是什么"、"接下来该干嘛"——依据全部来自这本账本的历史记录。

# 这也解释了 Agent 为什么有"上下文越长越聪明、也越贵"的特性：账本越厚，模型知道得越多，但每轮要读的字也越多。记忆、上下文压缩、子代理隔离，往后所有机制玩的都是这本账本。

# 机制二：tool_use_id 是回执单号
# 第 4 步里，每个结果都带着 tool_use_id: block.id。为什么？

# 因为模型一轮可以同时申请跑好几条命令。它发出的每个 tool_use 块都带一个唯一 id，就像银行转账的申请单号；你执行完，结果必须带着同一个单号塞回去，API 靠这个 id 配对"哪张回执对应哪笔申请"。

# id 对不上，API 直接报错。这是所有人第一次写 Agent 必踩的坑：辛辛苦苦执行完工具，结果塞回去的时候 id 忘了带或者带错了，前功尽弃。记住：tool_result 必须和 tool_use 一一配对。

# 机制三：循环什么时候停，代码说了不算，模型说了算
# 看第 3 步那个 if not tool_calls: return。

# 整个循环没有计数器、没有定时器、没有"最多跑 10 步"的设置。唯一的退出条件：模型这一轮没有申请任何工具。

# 模型不再要工具 = 模型认为任务做完了 = 循环结束。

# 这个设计初看朴素，细想是整个架构里最大胆的一笔。传统软件的流程控制权在程序员手里，而这里，"任务什么时候算完成"的定义权交给了模型。Agent 的自由度和边界感，全都浓缩在这一个 if 里。

# 第四块：命令行入口（10 行，带过）
# 最后补个壳，让它能在终端里交互：

if __name__ == "__main__":
    print("s01: Agent Loop")
    print("Enter a question, press Enter to send. Type q to quit.\n")

    history = []
    while True:
        try:
            query = input("agent >> ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() in ("q", "exit", ""):
            break
        history.append({"role": "user", "content": query})
        agent_loop(history)
        # 打印模型的最终文字回复
        for block in history[-1]["content"]:
            if getattr(block, "type", None) == "text":
                print(block.text)
        print()

# 注意 history 这个列表在主循环外面——你的每次提问都追加进同一本账本，所以它记得你上一句说了什么，多轮对话是免费自带的。

# 四块拼完，142 行。保存成 code.py。


s01: Agent Loop
Enter a question, press Enter to send. Type q to quit.

$ ls -la
'ls' is not recognized as an internal or external command,
operable program or batch file.
$ dir
Volume in drive G is 新加卷G（日常文件）
 Volume Serial Number is 2A29-3924

 Directory of g:\FromZeroToClaudeCode\Agentloop

2026/09/07  15:18    <DIR>          .
2026/09/07  14:40    <DIR>          ..
2026/0
当前目录 `g:\FromZeroToClaudeCode\Agentloop` 中有以下文件：

| 文件名 | 大小 |
|--------|------|
| `.env` | 224 字节 |
| `1.ipynb` | 4,018 字节 |
| `2.ipynb` | 2,107 字节 |
| `3.ipynb` | 11,578 字节 |
| `4.ipynb` | 119 字节 |
| `AgentLoop示意图.png` | 691,191 字节 |
| `code.py` | 0 字节 |

> 注：`code.py` 是一个空文件（0 字节）。

共 7 个文件。

$ dir /s | findstr "File(s)"
7 File(s)        710,273 bytes
               7 File(s)        710,273 bytes
从上一次 `dir` 的输出可以看到，**`AgentLoop示意图.png`** 体积最大，大小是 **691,191 字节**（约 675 KB）。



![](AgentLoop示意图.png)